In [47]:
# ===== Loading Data EKG =====
# ===== Import Libraries yang Digunakan =====

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import zscore

df = pd.read_csv("/Users/alstanlin/Desktop/CAT-NET for Arrhythmia Detection/Data Wearable EKG-Nano (Berhasil)/Wearable EKG-Nano15.csv")

# ===== Visualisasi Sinyal EKG =====

ekg = df.iloc[:, 2]
fs = 250

waktu = range(len(ekg))

# ===== Windowing Sinyal 10 Detik =====

def potong_jendela_10_detik(ekg, fs=250):
    ukuran_jendela = int(10 * fs)
    jendela_terkumpul = []

    for i in range(0,len(ekg) - ukuran_jendela + 1,ukuran_jendela):
        potongan = ekg[i : i + ukuran_jendela]
        jendela_terkumpul.append(potongan)
    return np.array(jendela_terkumpul)

matriks_jendela = potong_jendela_10_detik(ekg,fs=fs)

# ===== Bandpass Filter Sinyal EKG =====

def bandpass_filter(signal_data,lowcut=0.5,highcut=40,fs=250,order=3):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order,[low, high],btype='band')
    filtered_signal = filtfilt(b,a,signal_data)
    return filtered_signal

# ===== Denoising Sinyal EKG dengan Wavelet Soft-Thresholding (Motion Artifact Removal) =====

def denoise(signal_data, wavelet='db5', level=5):
    coeffs = pywt.wavedec(signal_data, wavelet=wavelet, level=level)
    coeffs[0] = np.zeros_like(coeffs[0])
    detail_highest = coeffs[-1]
    sigma = np.median(np.abs(detail_highest)) / 0.6745
    n = len(signal_data)
    universal_threshold = sigma * np.sqrt(2 * np.log(n))
    for i in range(1, len(coeffs)):  # Skip approximation (index 0)
        coeffs[i] = pywt.threshold(coeffs[i], value=universal_threshold, mode='soft')
    reconstructed = pywt.waverec(coeffs, wavelet=wavelet)
    return reconstructed[:len(signal_data)]

# ===== Proses Filtering, Denoising, dan Deteksi R-Peak =====

hasil_filter = []          # Sinyal SETELAH zscore (untuk analisis fitur)
hasil_filter_no_norm = []  # Sinyal SEBELUM zscore (untuk hitung SNR)
hasil_peak = []

for idx, window in enumerate(matriks_jendela):
    filtered = bandpass_filter(window)
    denoised = denoise(filtered)
    
    # Simpan SEBELUM zscore (untuk SNR)
    hasil_filter_no_norm.append(denoised)
    
    # Normalisasi untuk deteksi fitur
    normalized = zscore(denoised)
    hasil_filter.append(normalized)

    # Pan-Tompkins R-Peak Detection
    diff_signal = np.diff(normalized)
    squared_signal = diff_signal ** 2
    window_size = int(0.15 * fs)
    mwi_signal = np.convolve(squared_signal, np.ones(window_size)/window_size, mode='same')

    threshold_peak = 0.2 * np.max(mwi_signal)
    peaks_mwi, _ = find_peaks(mwi_signal, height=threshold_peak, distance=int(0.25 * fs))

    r_peaks = []
    search_window = int(0.1 * fs)
    for peak in peaks_mwi:
        start = max(0, peak - search_window)
        end = min(len(normalized), peak + search_window)
        local_peak = np.argmax(normalized[start:end])
        r_peak = start + local_peak
        r_peaks.append(r_peak)

    r_peaks = np.array(r_peaks)
    hasil_peak.append(r_peaks)

# ===== Deteksi Kondisi Normal atau Atrial Fibrillation (AFib) berdasarkan Fitur R-R Intervals per Window =====

hasil_label = []

for window_ke in range(len(hasil_peak)):
    r_peaks = hasil_peak[window_ke]
    if len(r_peaks) < 3:
        print(f"Window {window_ke+1}: RR interval tidak cukup")
        hasil_label.append("Tidak Valid")
        continue

    rr_intervals = np.diff(r_peaks) / fs
    mean_rr = np.mean(rr_intervals)
    mean_bpm = 60 / mean_rr
    sdnn = np.std(rr_intervals, ddof=1)
    diff_rr = np.diff(rr_intervals)
    rmssd = np.sqrt(np.mean(diff_rr**2))
    nn50 = np.sum(np.abs(diff_rr) > 0.05)
    pnn50 = (nn50 / len(diff_rr)) * 100
    if (rmssd > 0.10 and sdnn > 0.08 and pnn50 > 20):
        label = "AFib"
    else:
        label = "Normal"
    hasil_label.append(label)

    print(f"\nWindow {window_ke+1}")
    print(f"Label    : {label}")


Window 1
Label    : AFib

Window 2
Label    : AFib

Window 3
Label    : Normal

Window 4
Label    : AFib

Window 5
Label    : Normal

Window 6
Label    : AFib

Window 7
Label    : AFib

Window 8
Label    : Normal

Window 9
Label    : AFib

Window 10
Label    : AFib

Window 11
Label    : AFib

Window 12
Label    : AFib

Window 13
Label    : AFib

Window 14
Label    : AFib

Window 15
Label    : Normal

Window 16
Label    : AFib

Window 17
Label    : AFib

Window 18
Label    : Normal

Window 19
Label    : Normal

Window 20
Label    : Normal

Window 21
Label    : AFib

Window 22
Label    : AFib

Window 23
Label    : AFib

Window 24
Label    : AFib

Window 25
Label    : Normal

Window 26
Label    : Normal

Window 27
Label    : Normal

Window 28
Label    : Normal

Window 29
Label    : Normal

Window 30
Label    : Normal

Window 31
Label    : AFib

Window 32
Label    : Normal

Window 33
Label    : Normal

Window 34
Label    : Normal

Window 35
Label    : AFib

Window 36
Label    : Normal

Wi